In [7]:
from decimal import Decimal, getcontext, ROUND_HALF_UP
import yfinance as yf
import pandas as pd
import numpy as np
import math
from tqdm.auto import tqdm
from scipy.optimize import minimize
from functools import partial
from Data import Triple_Barrier_Labelel
from datasets import Dataset as HuggingfaceDataset
from transformers import (
    AutoTokenizer,
    AutoConfig,
    AutoModelForSequenceClassification
)
from sklearn.model_selection import GroupKFold
from torch.utils.data import DataLoader
import torch
from tqdm.auto import tqdm
from torch.optim.lr_scheduler import LambdaLR
from torch.utils.data import DataLoader, Dataset as torchDS
import emoji
import re
import string
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score, confusion_matrix

getcontext().prec = 10
getcontext().rounding = ROUND_HALF_UP  # financial round

In [8]:
LABEL_BULLISH = "Bullish"
LABEL_BEARISH = "Bearish"
LABEL_NEUTRAL = "Neutral"

param_grid = {
    'volatility_period': [8, 9, 10, 11, 12, 13, 14, 15],
    'upper_barrier_factor': [1.0, 1.1, 1.2, 1.3, 1.4, 1.5],
    'lower_barrier_factor': [1.0, 1.1, 1.2, 1.3, 1.4, 1.5],
    'vertical_barrier': [8, 9, 10, 11, 12, 13, 14, 15],
}

label_map = {
    LABEL_BEARISH: 0,
    LABEL_NEUTRAL: 1,
    LABEL_BULLISH: 2
}       

class Triple_Barrier_Labelel():
    def __init__(self):
        super()
        
    def calculate_barriers(self, price, volatility, Fu, Fl):
        upper = price + price * volatility * Fu
        lower = price - price * volatility * Fl
        return upper, lower
    
    def assign_labels(self, data, fu, fl, vt):
        barriers = None
        prev_label = None
        labels = []
        # loop over all data rows
        for index, day in data.iterrows():
            # check if barriers are calculated if not, calculate and set starting day
            if barriers == None:
                barriers = self.calculate_barriers(day["close"], day["volatility"], fu, fl)
                start_day = index
            days_last = abs((start_day - index).days)
            # If window is shorter than 3 days or only 2 days left to end of dataset, skip creating new labels
            if days_last < 3 and abs((start_day - data.index[-1]).days) > 2:
                continue
            # If price dont hit upper or lower barrier in specific vt period, assign neutral label
            if days_last == vt:
                labels.append({"lower_barrier": barriers[1], "higher_barrier": barriers[0], "start": start_day, "end": index, "label": LABEL_NEUTRAL, "prev_label": prev_label})
                barriers = None
                prev_label = LABEL_NEUTRAL
            # If price hit upper barrier assign bullish label
            elif day["high"] > barriers[0]:
                labels.append({"lower_barrier": barriers[1], "higher_barrier": barriers[0], "start": start_day, "end": index, "label": LABEL_BULLISH, "prev_label": prev_label})
                barriers = None
                prev_label = LABEL_BULLISH
            # If price hit lower barrier assign bearish label
            elif day["low"] < barriers[1]:
                labels.append({"lower_barrier": barriers[1], "higher_barrier": barriers[0], "start": start_day, "end": index, "label": LABEL_BEARISH, "prev_label": prev_label})
                barriers = None
                prev_label = LABEL_BEARISH
        return labels
    
    # Calculate volatility on specific time window, explicitly aligns each current date 
    # with the previous day's price, even if the index is non-continuous or has gaps (e.g., weekends)
    def get_daily_volatility(self, df, span = 14):
        prev_day_start = df.close.index.searchsorted(df.close.index - pd.Timedelta(days=1))
        prev_day_start = prev_day_start[prev_day_start > 0]
        prev_day_start = pd.Series(df.close.index[prev_day_start - 1], index=df.close.index[df.close.shape[0] - prev_day_start.shape[0]:])
        daily_returns = df.close.loc[prev_day_start.index] / df.close.loc[prev_day_start.values].values - 1
        return daily_returns.ewm(span=span).std()
    
    def map_data(self, labels, day, key):
        day = pd.to_datetime(day)
        for label in labels:
            start = pd.to_datetime(label['start'])
            end = pd.to_datetime(label['end'])
            if start <= day <= end:
                return label[key]
        return None  # If day is outside all defined windows
    
    def map_start_window(self, labels, day):
        day = pd.to_datetime(day)
        return any(pd.to_datetime(label['start']) == day for label in labels)
    
    def transform(self, df, vol, fu, fl, vt):
        copy = df.copy()
        copy["volatility"] = self.get_daily_volatility(copy, vol)
        labels = self.assign_labels(copy, fu, fl, vt)
        copy['label'] = copy.apply(lambda x: self.map_data(labels, x.name, 'label'), axis=1)
        copy['lower_barriers'] = copy.apply(lambda x: self.map_data(labels, x.name, 'lower_barrier'), axis=1)      
        copy['upper_barriers'] = copy.apply(lambda x: self.map_data(labels, x.name, 'higher_barrier'), axis=1)      
        copy['label'] = copy['label'].map(label_map)
        copy["previous_label"] = copy.apply(lambda x: self.map_data(labels, x.name, 'prev_label'), axis=1)      
        copy["window_start"] = copy.apply(lambda x: self.map_start_window(labels, x.name), axis=1)      
        copy['signals'] = copy["label"].apply(lambda x: 1 if x == 0 else 2 if x == 2 else 0)
        # Calculate the percentage changes for TP and SL
        copy['tp_stop'] = (copy['upper_barriers'] - copy['close']) / copy['close']
        copy['sl_stop'] = (copy['close'] - copy['lower_barriers']) / copy['close']
        # Replace negative values with 0
        copy['tp_stop'] = copy['tp_stop'].apply(lambda x: max(x, 0))
        copy['sl_stop'] = copy['sl_stop'].apply(lambda x: max(x, 0))
        return copy
    
    def optimize(self, data, num_starts = 10, initial_cash=100000, fee=0.006):
        optimized_params = []
        # Generate interval periods based on frequency and provided timespan
        intervals = pd.date_range(start=data.index.min(), end=data.index.max(), freq="6M")
        # For each period optimize vol, fu, fl, vt values by simulating returns with provided
        # values and optimized based on sharpe ratio
        for start, end in zip(intervals[:-1], intervals[1:]):
            best_sharpe_ratio, best_net_profit, best_params = -np.inf, -np.inf, None
            interval_df = data.loc[start:end]
            param_space = [
                param_grid['volatility_period'],
                param_grid['upper_barrier_factor'],
                param_grid['lower_barrier_factor'],
                param_grid['vertical_barrier'],
            ]
            
            def bounds_to_params(x):
                return {key: space[int(idx)] for key, space, idx in zip(param_grid.keys(), param_space, x)}
            
            def objective_wrapper(x):
                params = bounds_to_params(x)
                transformed = self.transform(interval_df, params['volatility_period'], params['upper_barrier_factor'], params['lower_barrier_factor'], params['vertical_barrier'])
                cr = Calculate_Returns(
                    initial_cash=initial_cash,
                    fee=fee,
                    prices=transformed['close'],
                    signals=transformed['signals'],
                    tp_stop=transformed['tp_stop'],
                    sl_stop=transformed['sl_stop']
                )
                cr.from_signals()
                return -cr.sharpe()
            bounds = [(0, len(space) - 1) for space in param_space]
            for _ in tqdm(range(num_starts), desc="Optimizing"):
                initial_guess = [np.random.randint(len(space)) for space in param_space]
                result = minimize(objective_wrapper, initial_guess, method='SLSQP', bounds=bounds)
                if result.success and -result.fun > best_sharpe_ratio:
                    best_sharpe_ratio = -result.fun
                    best_params = bounds_to_params(result.x)
            optimized_params.append({
                'start': start,
                'end': end,
                'params': best_params,
                'sharpe_ratio': best_sharpe_ratio,
            })
        return pd.DataFrame(optimized_params)

In [9]:
class Calculate_Returns():
    def __init__(self, initial_cash, fee, prices, signals, tp_stop, sl_stop):
        self.cash = Decimal(initial_cash) if initial_cash is not None else Decimal(0)
        self.investing_cash = Decimal(initial_cash) if initial_cash is not None else Decimal(0)
        self.fee = Decimal(fee)
        self.prices = prices.apply(Decimal)
        self.signals = signals
        self.tp_stop = tp_stop.apply(Decimal)
        self.sl_stop = sl_stop.apply(Decimal)
        self.context = 0 # 0-hold, 1-short, 2-long
        self.returns = [Decimal(0)]
        self.records = []
        self.entry_data = {
            "price": None,
            "date": None,
            "tp": None,
            "sl": None,
            "size": Decimal(1)
        }
        
    def add_record(self, direction, exit_price, exit_date):
        if direction != "Long" and direction != "Short":
            print('Error. Direction must be "Long" or "Short"')
            return 0
        entry_value = self.entry_data['size'] * self.entry_data['price']
        exit_value = self.entry_data['size'] * exit_price
        entry_fees = self.fee * self.entry_data['size'] * self.entry_data['price']
        exit_fees = self.fee * self.entry_data['size'] * exit_price
        total_fees = entry_fees + exit_fees
        pnl = exit_value - entry_value - total_fees if direction == "Long" else entry_value - exit_value - total_fees
        self.records.append({
            "Size": self.entry_data['size'],
            "Entry Timestamp": self.entry_data['date'],
            "Avg Entry Price": self.entry_data['price'],
            "Entry fees": entry_fees,
            "Exit Timestamp": exit_date,
            "Avg Exit Price": exit_price,
            "Exit Fees": exit_fees,
            "PnL": pnl,
            "Return": pnl / entry_value if entry_value != 0 else Decimal(0),
            "Direction": direction,
            "Status": "Closed",
        })
        
    def sharpe(self, period = 365):
        returns_array = np.array([float(r) for r in self.returns])
        if returns_array.std() == 0:
            return 0
        return (returns_array.mean() / returns_array.std()) * math.sqrt(period)
        
    def is_position_opened(self):
        return (
            all(self.entry_data[k] is not None for k in self.entry_data if k != "size")
            and self.entry_data["size"] == Decimal(1)
        )
    
    def save_entry_position(self, price, date, tp, sl, size):
        self.entry_data.update({
            "price": price,
            "date": date,
            "tp": tp,
            "sl": sl,
            "size": size
        })
    
    def drop_entry_position(self):
        self.entry_data = {
            key: None if key != "size" else Decimal(1)
            for key in self.entry_data
        }
        self.context = 0

    def get_short_return_value(self, prev_price, curr_price):
        return prev_price - curr_price

    def get_long_return_value(self, prev_price, curr_price):
        return (curr_price - prev_price) * self.entry_data['size']

    def get_fee_value(self, curr_price):
        return -(curr_price * self.fee * self.entry_data['size'])
    
    def update_cash(self, ret, fee):
        self.cash += ret + fee
    
    def add_return(self, ret, fee):
        if self.cash == 0:
            self.returns.append(Decimal(0))
        else:
            self.returns.append((ret + fee)/self.cash)
        
    def open_short(self, i, closing_ret = Decimal(0), closing_fee = Decimal(0)):
        # Abort operation if there is no cash
        # If there is a cash, update context, then save entry position
        # Calculate fee then add previous position returns and fees if exits,
        # Finally combine all costs to retrieve return and update cash
        if self.investing_cash < Decimal(0):
            return None
        self.context = 1 # Change context to short
        self.save_entry_position(self.prices[i], self.prices.index[i], self.tp_stop[i], self.sl_stop[i], Decimal(1))
        fee_opening = self.get_fee_value(self.prices[i])
        self.add_return(fee_opening, closing_ret + closing_fee)
        self.update_cash(fee_opening, closing_ret + closing_fee)
    
    def close_short(self, i):
        # Update investing cash, calculate return and fee then save to logs,
        # Clean open position then return calculated return and fee
        self.investing_cash += self.entry_data['price'] - self.prices[i] - (self.prices[i] * self.fee) - (self.entry_data['price'] * self.fee)
        closing_ret = self.get_short_return_value(self.prices[i - 1], self.prices[i])
        closing_fee = self.get_fee_value(self.prices[i])
        self.add_record("Short", self.prices[i], self.prices.index[i])
        self.drop_entry_position()
        return closing_ret, closing_fee
    
    def open_long(self, i, closing_ret = Decimal(0), closing_fee = Decimal(0)):
        # Retrieve size of asset possible to purchase based on investing cash
        # update context, then save entry position
        # Calculate fee then add previous position returns and fees if exits,
        # Finally combine all costs to retrieve return and update cash
        position_size = Decimal(min(Decimal(1), (self.investing_cash - (self.investing_cash * self.fee)) / self.prices[i]))
        # No cash, so aboart
        if position_size <= Decimal(0):
            return None
        # Purchase 1 asset
        elif position_size == Decimal(1):
            self.investing_cash -= (self.prices[i] - (self.prices[i] * self.fee))
        # Purchase part of asset, more than 0.0 but less than 1.0
        else:
            self.investing_cash = Decimal(0)
        self.context = 2 # Change context to long
        self.save_entry_position(self.prices[i], self.prices.index[i], self.tp_stop[i], self.sl_stop[i], position_size)
        fee_opening = self.get_fee_value(self.prices[i])
        self.add_return(fee_opening, closing_ret + closing_fee)
        self.update_cash(fee_opening, closing_ret + closing_fee)
    
    def close_long(self, i):
        # Update investing cash, calculate return and fee then save to logs,
        # Clean open position then return calculated return and fee
        self.investing_cash += (self.entry_data['size'] * self.prices[i]) - (self.entry_data['size'] * self.prices[i] * self.fee)
        closing_ret = self.get_long_return_value(self.prices[i - 1], self.prices[i])
        closing_fee = self.get_fee_value(self.prices[i])
        self.add_record("Long", self.prices[i], self.prices.index[i])
        self.drop_entry_position()
        return closing_ret, closing_fee
        
    def check_tp_sl_barriers(self, i):
        if self.is_position_opened():
            if self.context == 1:
                # If price hits Take Profit (TP) price or Stop Loss (TL) price, close short position 
                if self.prices[i] <= self.entry_data['price'] * (Decimal(1) - self.entry_data['tp']) or \
                self.prices[i] >= self.entry_data['price'] * (Decimal(1) + self.entry_data['sl']):
                    closing_ret, closing_fee = self.close_short(i)
                    self.add_return(closing_ret, closing_fee)
                    self.update_cash(closing_ret, closing_fee)
                    return True
            else:
                # If price hits Take Profit (TP) price or Stop Loss (TL) price, close long position 
                if self.prices[i] >= self.entry_data['price'] * (Decimal(1) + self.entry_data['tp']) or \
                self.prices[i] <= self.entry_data['price'] * (Decimal(1) - self.entry_data['sl']):
                    closing_ret, closing_fee = self.close_long(i)
                    self.add_return(closing_ret, closing_fee)
                    self.update_cash(closing_ret, closing_fee)
                    return True
        return False
    
    def from_signals(self):
        for i in range(1, len(self.prices)):
            action = self.signals[i]
            if self.check_tp_sl_barriers(i):
                continue
            # --- Action Logic ---
            if action == 0:  # Hold
                if self.context == 0: # Keep holding
                    self.returns.append(Decimal(0))
                elif self.context == 1:  # Hold while short so Continue short
                    ret = self.get_short_return_value(self.prices[i - 1], self.prices[i])
                    self.add_return(ret, Decimal(0))
                    self.update_cash(ret, Decimal(0))
                elif self.context == 2:  # Hold while long so Continue long
                    ret = self.get_long_return_value(self.prices[i - 1], self.prices[i])
                    self.add_return(ret, Decimal(0))
                    self.update_cash(ret, Decimal(0))
            elif action == 1:  # Short
                if self.context == 0:  # Opening short
                    self.open_short(i)
                elif self.context == 1:  # Continue short
                    ret = self.get_short_return_value(self.prices[i - 1], self.prices[i])
                    self.add_return(ret, Decimal(0))
                    self.update_cash(ret, Decimal(0))
                elif self.context == 2:  # Close long, open short
                    closing_ret, closing_fee = self.close_long(i)
                    self.open_short(i, closing_ret, closing_fee)
            elif action == 2:  # Long
                if self.context == 0:  # Opening long
                    self.open_long(i)
                elif self.context == 1:  # Close short, open long
                    closing_ret, closing_fee = self.close_short(i)
                    self.open_long(i, closing_ret, closing_fee)
                elif self.context == 2:  # Continue long
                    ret = self.get_long_return_value(self.prices[i - 1], self.prices[i])
                    self.add_return(ret, Decimal(0))
                    self.update_cash(ret, Decimal(0))
    def test(self):
        data = yf.download("BTC-USD", start="2020-01-01", end="2025-01-01").reset_index()
        data.columns = data.columns.get_level_values(0)
        data = data.rename(columns={"Date": 'date', 'Close': 'close', 'Low': 'low', 'High': 'high', 'Open': 'open'})
        data["date"] = pd.to_datetime(data["date"])
        data = data.set_index("date")
        tbl = Triple_Barrier_Labelel(data)
        transformed = tbl.transform(data, 15, 1.5, 1.1, 12)
        transformed = pd.DataFrame(transformed)
        cr = Calculate_Returns(
            initial_cash=10000,
            fee=0.005,
            prices=transformed['close'],
            signals=transformed['signals'],
            tp_stop=transformed['tp_stop'],
            sl_stop=transformed['sl_stop']
        )
        cr.from_signals()
        return tbl.optimize(num_starts=10, initial_cash=100000, fee=0.006)

In [ ]:
class Data_Preprocessing():
    def __init__(self):
        super(self)
    
    #Load stock data from specific period and apply basic transformations by setting date as index
    def load_stock_data(self, stock_name = 'BTC-USD', starting_date, ending_date):
        price_df = yf.download(stock_name, start=starting_date, end=ending_date).reset_index()
        price_df.columns = price_df.columns.get_level_values(0)
        price_df = price_df.rename(columns={'Date': 'timestamp', 'Close': 'close', 'Low': 'low', 'High': 'high', 'Open': 'open', 'Volume': 'volume'})
        price_df = price_df.set_index("timestamp")
        price_df.index = pd.to_datetime(price_df.index)
        return price_df
    
    #Apply optimize function and add optimized labels for dataset with prompting label distribution (bearish/bullish/neutral)
    def add_optimized_labels(self, price_df):
        tbl = Triple_Barrier_Labelel()
        optimized_params_df = tbl.optimize(price_df, num_starts = 5)
        labeled_df = pd.DataFrame()
        for _, row in optimized_params_df.iterrows():
            start, end, params = row['start'], row['end'], row['params']
            interval_df = price_df.loc[start:end]
            transformed = tbl.transform(df=interval_df, vol=params['volatility_period'], fu=params['upper_barrier_factor'], fl=params['lower_barrier_factor'], vt=params['vertical_barrier'])
            labeled_df = pd.concat([labeled_df, transformed])
        print("Label distribution: ")
        print(labeled_df.label.value_counts(), optimized_params_df.sharpe_ratio)
        return labeled_df
    
    #Load tweet data from file and filter by id (Tweets only related to specific tag), then sort and 
    #add date as index
    def load_tweet_data(self, data_path = "phd.tweets.csv", tag_id = "67082fb60891532d63a3ed67"):
        tweet_df = pd.read_csv(data_path)
        tweet_df = tweet_df[tweet_df["tag_id"] == tag_id].sort_values(by='date')[["content", "date"]]
        tweet_df = tweet_df.rename(columns={'date': 'timestamp'})
        tweet_df = tweet_df.set_index("timestamp")
        tweet_df.index = pd.to_datetime(tweet_df.index, unit="ns")
        tweet_df["day"] = tweet_df.index
        tweet_df["day"] = tweet_df.day.apply(lambda x: x.date())
        tweet_df.index = tweet_df.index.normalize()
        tweet_df.index = tweet_df.index.date
        tweet_df = tweet_df.sort_index()
        return tweet_df
    
    #Add target label as simply next day label
    def add_target_label(self, labeled_df):
        labeled_df.rename(columns={'label': 'previous_label'}, inplace=True)
        labeled_df["next_day_label"] = labeled_df.previous_label.shift(-1)
        labeled_df["next_day_window_start"] = labeled_df.window_start.shift(-1)
        labeled_df.loc[labeled_df.iloc[0].name, 'next_day_window_start'] = True
        return labeled_df
    
    # Calculate RSI with anilla RSI formula based on gains/losses smoothed via .rolling().mean()
    # — a simplified RSI approximation (not EMA-smoothing as in Wilder’s method) .
    # The output are descriptive labels based on thresholds (default 30/70)
    def calculate_rsi(self, close_series, length, threshold=(30, 70)):
        delta = close_series.diff()
        gain = (delta.where(delta > 0, 0)).rolling(window=length).mean()
        loss = (-delta.where(delta < 0, 0)).rolling(window=length).mean()
        rs = gain / loss
        rsi = 100 - (100 / (1 + rs))
        description = pd.Series(index=close_series.index, dtype='object')
        # Apply dynamic thresholds if provided
        if threshold is not None:
            lower_threshold, upper_threshold = threshold
            description[rsi > upper_threshold] = 'bearish'
            description[rsi < lower_threshold] = 'bullish'
            description[(rsi <= upper_threshold) & (rsi >= lower_threshold)] = 'neutral'

        return rsi, description

    # Computes rolling percentage returns then defines adaptive boundaries: bullish if return > +0.4 * rolling_std
    # bearish if return < −0.5 * rolling_std, and neutral otherwise making volatility-aware (adaptive thresholds)
    def calculate_returns(self, close_series, length, std_coef = (-0.5, 0.4)):
        returns = close_series.pct_change()
        neg_std_coef, pos_std_coef = std_coef
        rolling_std = returns.rolling(window=length).std()
        rolling_neg_std_effect = rolling_std * neg_std_coef
        rolling_pos_std_effect = rolling_std * pos_std_coef
        # Descriptive values
        description = pd.Series(index=close_series.index, dtype='object')
        description[returns < rolling_neg_std_effect] = 'bearish'
        description[returns > rolling_pos_std_effect] = 'bullish'
        description[(returns <= rolling_pos_std_effect) & (returns >= rolling_neg_std_effect)] = 'neutral'
        return returns, description
    
    #Add RSI and ROC technical indicators
    def add_technical_indicators(self, labeled_df):
        _, labeled_df["RSI"] = calculate_rsi(labeled_df.close, 8, (30, 70))
        _, labeled_df["ROC"] = calculate_returns(labeled_df.close, 8)
        print("ROC and RSI label distibution: ")
        print(labeled_df["RSI"].value_counts())
        labeled_df["ROC"].value_counts()
        return labeled_df

    #Merge by date textual tweet data and labeled stock data 
    def merge_data(self, labeled_df, tweet_df):
        merged_df = tweet_df.merge(
            labeled_df[["next_day_label", 'next_day_window_start', 'previous_label', 'ROC', "RSI"]], left_index=True, right_index=True, how="left"
        )
        merged_df.dropna(inplace=True)
        print("Next day label distribution: ")
        merged_df.next_day_label.value_counts()
        return merged_df
    
    def undersample_tweets(self, merged_df):
        # Count the number of tweets for each trend
        trend_counts = merged_df['next_day_label'].value_counts()
        # Identify the minority class
        minority_class = trend_counts.idxmin()
        minority_count = trend_counts.min()
        # Initialize an empty DataFrame to store the undersampled data
        undersampled_df = pd.DataFrame()
        # Iterate over the trends
        for trend in merged_df['next_day_label'].unique():
            # If this is the minority class, add all tweets to the undersampled data
            if trend == minority_class:
                undersampled_df = pd.concat([undersampled_df, merged_df[merged_df['next_day_label'] == trend]])
            else:
                # Otherwise, randomly select a subset of tweets equal to the minority count
                subset = merged_df[merged_df['next_day_label'] == trend].sample(minority_count)
                undersampled_df = pd.concat([undersampled_df, subset])
        return undersampled_df
    
    # Make sure all label categories count is the same and balanced
    def undersample_label_data(self, merged_df):
        balanced_df = undersample_tweets(merged_df)
        print("Next day label distribution after undersample: ")
        print(balanced_df.next_day_label.value_counts())
        return balanced_df
    
    # Generate LLM understandable prompt structure for training eg: "previous label: bullish, roc: bearish, rsi: bullish, tweet: Bitcoin is on the rise!"
    def generate_tweet_prompts(df, include_previous_label=True, include_roc=True, include_rsi=True):
        def id_to_label(x):
            return "bullish" if x == 2 else "neutral" if x == 1 else "bearish"

        prompts = []
        for _, row in df.iterrows():
            prompt_parts = []
            if include_previous_label:
                prompt_parts.append(f"previous label: {id_to_label(row['previous_label'])}")
            if include_roc:
                prompt_parts.append(f"roc: {row['ROC']}")
            if include_rsi:
                prompt_parts.append(f"rsi: {row['RSI']}")
            prompt_parts.append(f"tweet: {row['text']}")
            prompts.append(" ".join(prompt_parts))

        return prompts
    
    # Tokenize data to be fed into a BERT-style model like BertForSequenceClassification.
    # Eg. "Stocks are rising today." -> { input_ids: [CLS, stocks, are, rising, today, ., PAD, ..., PAD],
    # attention_mask: [1, 1, 1, 1, 1, 1, 0, ..., 0] }
    def tokenize(tokenizer, dataset):
        # Tokenize the text field in the dataset
        def tokenize_function(tokenizer, item):
            # Tokenize the text and return only the necessary fields
            encoded = tokenizer(item["text"], padding="max_length", truncation=True, max_length=512)
            return {"input_ids": encoded["input_ids"], "attention_mask": encoded["attention_mask"], "label": item["label"]}
        # tokenizing the dataset text to be used in train and test loops
        partial_tokenize_function = partial(tokenize_function, tokenizer)
        tokenized_datasets = dataset.map(partial_tokenize_function, batched=True)
        return tokenized_datasets

    # Apply text cleaning and preprocessing to drop not useful data
    def transform_dataset(self, dataset, transformations):
        def apply_transformations(text, transformations):
            if "REMOVE_USERNAMES" in transformations:
                text = re.sub(r"@\w+", "", text)
            if "REMOVE_URLS" in transformations:
                text = re.sub(r"(?:https?://|www\.)\S+\.\S+", "", text)
            if "REMOVE_PUNCTUATION_MARKS" in transformations:
                remove_pun_pattern_1 = r"(?<!\d)\.(?!\d)|[^\w\s.']"
                remove_pun_pattern_2 = r"'"
                remove_pun_pattern_3 = r"\s\s+"
                sub1 = re.sub(remove_pun_pattern_1, " ", text)
                sub2 = re.sub(remove_pun_pattern_2, "", sub1)
                text = re.sub(remove_pun_pattern_3, " ", sub2)
            if "REMOVE_PUNCTUATION_WITH_EXCLUDE" in transformations:
                exclude = set(string.punctuation)
                for char in ['!','?','%','$','&']:
                    exclude.remove(char)
                return ''.join(ch for ch in text if ch not in exclude)
            if "TEXT_TO_LOWER" in transformations:
                text = text.lower()
            if "REMOVE_N" in transformations:
                text = text.replace("\\n", "").replace("\n", "")
            if "REMOVE_SHORT_WORDS" in transformations:
                remove_short_pattern = r'\b\w{1,2}\b'
                text = re.sub(remove_short_pattern, '', text)
            if "REMOVE_ENDING_HASHTAGS" in transformations:
                pattern = r'([#$@\$]\w+)(?=(\s[#$@\$]\w+)*\s*$)'
                text = re.sub(pattern, '', text)
            if "REMOVE_HASHTAGS" in transformations:
                text = re.findall(r"#(\w+)", text)
            if "REMOVE_HEX" in transformations:
                text = re.sub(r'\b0x[a-fA-F0-9]{6,}\b', '', text)
            if "REMOVE_EMOTICONS" in transformations:
                emoji_pattern = re.compile(
                    "[" 
                    u"\U0001F600-\U0001F64F"  # Emoticons
                    u"\U0001F300-\U0001F5FF"  # Symbols & pictographs
                    u"\U0001F680-\U0001F6FF"  # Transport & map
                    u"\U0001F1E0-\U0001F1FF"  # Flags
                    u"\U00002700-\U000027BF"  # Dingbats
                    u"\U0001F900-\U0001F9FF"  # Supplemental Symbols & Pictographs (includes 🤝)
                    u"\U00002600-\U000026FF"  # Misc symbols (e.g. ☀️☂️)
                    u"\U00002B00-\U00002BFF"  # Arrows etc.
                    u"\U0001FA70-\U0001FAFF"  # Symbols and Pictographs Extended-A
                    "]+", flags=re.UNICODE
                )
                text = emoji_pattern.sub(r'', text)
            if "REMOVE_SPACES" in transformations:
                text = re.sub(r"\s{2,}", " ", text).strip()
            if "REPLACE_WITH_BTC" in transformations:
                text = re.sub(r"Bitcoin|bitcoin|btc|BitCoin", "BTC", text)
            return text
        def transform_batch(batch):
            return {"text": [apply_transformations(text, transformations) for text in batch["text"]]}
        return dataset.map(transform_batch, batched=True)
    
    # Apply preprocessing and tokenization of data
    def preprocess_and_tokenize(self, labeled_ds, transformations = ["TEXT_TO_LOWER", "REMOVE_URLS", "REMOVE_USERNAMES", "REMOVE_PUNCTUATION_WITH_EXCLUDE", "REPLACE_WITH_BTC"]):
        labeled_ds = transform_dataset(labeled_ds, transformations)
        labeled_ds = labeled_ds.class_encode_column('label')
        tokenizer = AutoTokenizer.from_pretrained("ProsusAI/finbert")
        labeled_ds = tokenize(
            tokenizer, labeled_ds
        )
        return labeled_ds
    
    def test(self):
        price_df = self.load_stock_data("BTC-USD", "2020-01-01", "2025-01-01")
        labeled_df = self.add_optimized_labels(price_df)
        labeled_df = self.add_target_label(labeled_df)
        labeled_df = self.add_technical_indicators(labeled_df)
        tweet_df = self.load_tweet_data()
        merged_df = self.merge_data(labeled_df, tweet_df)
        balanced_df = self.undersample_label_data(merged_df)
        balanced_df["text"] = self.generate_tweet_prompts(balanced_df)
        balanced_df = balanced_df.dropna()
        balanced_df = balanced_df.sort_index()
        balanced_df["label"] = balanced_df.next_day_label
        labeled_ds = HuggingfaceDataset.from_pandas(balanced_df[["text", "label"]])
        return self.preprocess_and_tokenize(labeled_ds)